In [1]:
# run this by itself to run r
%load_ext rpy2.ipython
# -i "import" to make it readable in R
# -o "output" to make it readable in python

Error importing in API mode: ImportError('On Windows, cffi mode "ANY" is only "ABI".')
Trying to import in ABI mode.


In [2]:
# Imports 

import numpy as np 
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

import optuna

In [3]:
import pandas as pd
df_test = pd.read_csv(
    "C:/Users/tyler/OneDrive/Documents/GitHub/work_to_show/Kaggle/Titanic/test.csv"
)
df_train = pd.read_csv(
    "C:/Users/tyler/OneDrive/Documents/GitHub/work_to_show/Kaggle/Titanic/train.csv"
)

In [4]:
# One Hot Encoding Categorical Stuff

class FeatureEngineering(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        
        X[["Cabin"]] = X[["Cabin"]].isna().astype(int)
        X["Sex"] = X["Sex"].map({"male": 0, "female": 1})

        embark_mapping = {"S": 1, "C":2, "Q":3}
        X["Embarked"] = X["Embarked"].map(embark_mapping)
        X["FamilySize"] = X["SibSp"] + X["Parch"] + 1
        X["IsAlone"] = (X["FamilySize"] == 1).astype(int)
        X["FarePerPerson"] = X["Fare"] / X["FamilySize"]
        
        X["Title"] = X["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
        X["Title"] = X["Title"].replace({
            "Mlle": "Miss",
            "Ms": "Miss",
            "Mme": "Mrs"
        })

        X["Title"] = X["Title"].replace([
            "Lady","Countess","Capt","Col","Don","Dr","Major","Rev","Sir","Jonkheer","Dona"
        ], "Rare")
        
        X["Title"] = X["Title"].map({"Mr": 0, "Mrs": 1, "Master": 2, "Miss": 3, "Rare": 4})

        X["FamilySizeGroup"] = pd.cut(X["FamilySize"], bins=[0,1,4,10], labels=[0,1,2])
        X["AgeGroup"] = pd.cut(X["Age"], bins=[0,12,20,40,60,100], labels=[0,1,2,3,4])
        X["FareGroup"] = pd.qcut(X["Fare"], 4, labels=False)

        return X[["Cabin", "Sex", "Embarked", "Fare", "Title", "Age", "FamilySize", "IsAlone", "FarePerPerson", "Pclass", "SibSp", "Parch"]]

In [5]:
# optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=69)
    
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.5),
        "n_estimators": trial.suggest_int("n_estimators", 1, 500),
        "subsample": trial.suggest_float("subsample", 0.0, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.0, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 5),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 5),
    }
    
    pipeline = Pipeline([
        ("feature_eng", FeatureEngineering()),
        ("imputer", SimpleImputer(strategy="mean")),
        ("model", XGBClassifier(**params, eval_metric='logloss'))
    ])
    
    return cross_val_score(pipeline, df_train, y, cv=skf, scoring="accuracy").mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print(study.best_params)

[I 2026-03-19 23:27:49,654] A new study created in memory with name: no-name-857eccdb-a136-4c5d-b633-3b4c04556758
[W 2026-03-19 23:27:49,662] Trial 0 failed with parameters: {'max_depth': 3, 'learning_rate': 0.3662692837761902, 'n_estimators': 392, 'subsample': 0.0643863362113607, 'colsample_bytree': 0.25056758895274744, 'gamma': 3.1202216262836053, 'reg_alpha': 1.3911774731081787, 'reg_lambda': 4.024075554912836} because of the following error: NameError("name 'y' is not defined").
Traceback (most recent call last):
  File "C:\Users\tyler\anaconda3\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\tyler\AppData\Local\Temp\ipykernel_1832\1362754248.py", line 24, in objective
    return cross_val_score(pipeline, df_train, y, cv=skf, scoring="accuracy").mean()
                                               ^
NameError: name 'y' is not defined
[W 2026-03-19 23:27:49,669] Trial 0 failed with value None.


NameError: name 'y' is not defined

In [41]:
model = XGBClassifier(**study.best_params)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=69)


y = df_train["Survived"]
X_train, X_train_test, y_train, y_test = train_test_split(df_train, y, test_size=0.2, random_state=69)

pipeline = Pipeline([
    ("feature_eng", FeatureEngineering()),
    ("imputer", SimpleImputer(strategy="mean")),
    ("model", model)
])

pipeline.fit(X_train, y_train)

train_pred = pipeline.predict(X_train)
test_pred = pipeline.predict(X_train_test)

print(f"Train Accuracy: {accuracy_score(y_train, train_pred)}")
print(f"Test Accuracy: {accuracy_score(y_test, test_pred)}")
print(f"CM:\n {confusion_matrix(y_test, test_pred)}")
print(f"Classification Report:\n {classification_report(y_test, test_pred)}")
print(f"CV Score:\n {cross_val_score(pipeline, X_train_test, y_test, cv=skf).mean()}")

 [1] "PassengerId" "Survived"    "Pclass"      "Name"        "Sex"        
 [6] "Age"         "SibSp"       "Parch"       "Ticket"      "Fare"       
[11] "Cabin"       "Embarked"   


C:\Users\tyler\anaconda3\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "Cabin". Fall back to string conversion. The error is: Series can only be of one type, or None (and here we have <class 'float'> and <class 'str'>). If happening with a pandas DataFrame the method infer_objects() will normalize data types before conversion.
  warnings.warn('Error while trying to convert '


In [42]:
# clean data

def preprocess(df):
    df = df.copy()
    
    def normalize_name(x):
        return " ".join([v.strip(",()[].\"'") for v in x.split(" ")])
    
    def ticket_number(x):
        return x.split(" ")[-1]
        
    def ticket_item(x):
        items = x.split(" ")
        if len(items) == 1:
            return "NONE"
        return "_".join(items[0:-1])
    
    df["Name"] = df["Name"].apply(normalize_name)
    df["Ticket_number"] = df["Ticket"].apply(ticket_number)
    df["Ticket_item"] = df["Ticket"].apply(ticket_item)                     
    return df
    
preprocessed_df_train = preprocess(df_train)
preprocessed_df_test = preprocess(df_test)

preprocessed_df_train.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Ticket_number,Ticket_item
0,1,0,3,Braund Mr Owen Harris,male,22.0,1,0,A/5 21171,7.2500,NaN,S,21171,A/5
1,2,1,1,Cumings Mrs John Bradley Florence Briggs Thayer,female,38.0,1,0,PC 17599,71.2833,C85,C,17599,PC
2,3,1,3,Heikkinen Miss Laina,female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,3101282,STON/O2.
3,4,1,1,Futrelle Mrs Jacques Heath Lily May Peel,female,35.0,1,0,113803,53.1000,C123,S,113803,NONE
4,5,0,3,Allen Mr William Henry,male,35.0,0,0,373450,8.0500,NaN,S,373450,NONE


In [43]:
from sklearn.base import BaseEstimator, TransformerMixin
class FeatureEngineering(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        
        X[["Cabin"]] = X[["Cabin"]].isna().astype(int)
        X["Sex"] = X["Sex"].map({"male": 0, "female": 1}) # hot code male and female

        embark_mapping = {"S": 1, "C":2, "Q":3}
        X["Embarked"] = X["Embarked"].map(embark_mapping)
        X["FamilySize"] = X["SibSp"] + X["Parch"] + 1
        X["IsAlone"] = (X["FamilySize"] == 1).astype(int)
        X["FarePerPerson"] = X["Fare"] / X["FamilySize"]
        
        X["Title"] = X["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
        X["Title"] = X["Title"].replace({
            "Mlle": "Miss",
            "Ms": "Miss",
            "Mme": "Mrs"
        })

        X["Title"] = X["Title"].replace([
            "Lady","Countess","Capt","Col","Don","Dr","Major","Rev","Sir","Jonkheer","Dona"
        ], "Rare")
        
        X["Title"] = X["Title"].map({"Mr": 0, "Mrs": 1, "Master": 2, "Miss": 3, "Rare": 4})

        X["FamilySizeGroup"] = pd.cut(X["FamilySize"], bins=[0,1,4,10], labels=[0,1,2])
        X["AgeGroup"] = pd.cut(X["Age"], bins=[0,12,20,40,60,100], labels=[0,1,2,3,4])
        X["FareGroup"] = pd.qcut(X["Fare"], 4, labels=False)

        return X[["Cabin", "Sex", "Embarked", "Fare", "Title", "Age", "FamilySize", "IsAlone", "FarePerPerson", "Pclass", "SibSp", "Parch"]]

In [44]:
# optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=69)
    
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.5),
        "n_estimators": trial.suggest_int("n_estimators", 1, 500),
        "subsample": trial.suggest_float("subsample", 0.0, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.0, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 5),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 5),
    }
    
    pipeline = Pipeline([
        ("feature_eng", FeatureEngineering()),
        ("imputer", SimpleImputer(strategy="mean")),
        ("model", XGBClassifier(**params, eval_metric='logloss'))
    ])
    
    return cross_val_score(pipeline, df_train, y, cv=skf, scoring="accuracy").mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print(study.best_params)

[I 2026-03-19 23:25:06,844] A new study created in memory with name: no-name-684275de-5a29-4dfd-ac89-acb26a0d2395
[W 2026-03-19 23:25:06,848] Trial 0 failed with parameters: {'max_depth': 3, 'learning_rate': 0.3767078477825785, 'n_estimators': 409, 'subsample': 0.2041815714039894, 'colsample_bytree': 0.2752754321259212, 'gamma': 0.3456331742989843, 'reg_alpha': 1.4437925812691144, 'reg_lambda': 3.7035086615533714} because of the following error: NameError("name 'y' is not defined").
Traceback (most recent call last):
  File "C:\Users\tyler\anaconda3\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\tyler\AppData\Local\Temp\ipykernel_24924\1362754248.py", line 24, in objective
    return cross_val_score(pipeline, df_train, y, cv=skf, scoring="accuracy").mean()
                                               ^
NameError: name 'y' is not defined
[W 2026-03-19 23:25:06,856] Trial 0 failed with value None.


NameError: name 'y' is not defined